# ReplicaLab V2 Training Notebook

This notebook is the judged-path driver for the new training modules under `replicalab/training/`.

It keeps the Unsloth / TRL flow from the reference GRPO notebook, but moves the real project logic into reusable Python modules so the same code can run in Colab, on an H100 VM, or inside a Northflank GPU job.

Planned flow:
1. Load frozen evidence packs from the local 50-paper corpus.
2. Preview Scientist GRPO and Lab Manager SFT datasets.
3. Run dry-run training plans first.
4. Flip the real training flag only after the previews look correct.
5. Evaluate on fixed seeds and export artifacts.


In [ ]:
%%capture
!pip install --upgrade -qqq uv
!uv pip install -e .
!uv pip install unsloth unsloth_zoo trl vllm datasets matplotlib


In [ ]:
import os

from replicalab.training import (
    ArtifactLayout,
    LabManagerSFTConfig,
    ScientistGRPOConfig,
    build_default_evaluation_cases,
    build_lab_manager_sft_examples,
    build_scientist_prompt_examples,
    evaluate_policy,
    load_frozen_evidence_packs,
    preview_lab_manager_training,
    preview_scientist_training,
    train_lab_manager_sft,
    train_scientist_grpo,
)
from replicalab.agents import build_baseline_scientist_action

REPLICALAB_URL = os.environ.get("REPLICALAB_URL", "https://ayushozha-replicalab.hf.space")
REPLICALAB_PERSIST_ROOT = os.environ.get("REPLICALAB_PERSIST_ROOT", "/workspace/persist/replicalab")
RUN_REAL_TRAINING = False
SCIENTIST_MODEL = "Qwen/Qwen3-8B"
LAB_MANAGER_MODEL = "Qwen/Qwen3-8B"


In [ ]:
evidence_packs = load_frozen_evidence_packs()
print(f"loaded evidence packs: {len(evidence_packs)}")

scientist_examples = build_scientist_prompt_examples(
    seeds=[0, 1],
    templates=["math_reasoning", "ml_benchmark", "finance_trading"],
    difficulties=["easy"],
    evidence_packs=evidence_packs,
)
lab_manager_examples = build_lab_manager_sft_examples(
    seeds=[0],
    templates=["ml_benchmark", "finance_trading"],
    difficulties=["easy"],
    evidence_packs=evidence_packs,
)

print("scientist prompt rows:", len(scientist_examples))
print("lab manager rows:", len(lab_manager_examples))
scientist_examples[0]


In [ ]:
scientist_layout = ArtifactLayout.create(
    run_name="scientist-preview",
)
scientist_config = ScientistGRPOConfig(
    model_name=SCIENTIST_MODEL,
    train_seeds=list(range(8)),
    max_steps=300,
)
scientist_plan = preview_scientist_training(scientist_config, layout=scientist_layout)
scientist_plan


In [ ]:
lab_manager_layout = ArtifactLayout.create(
    run_name="lab-manager-preview",
)
lab_manager_config = LabManagerSFTConfig(
    model_name=LAB_MANAGER_MODEL,
    train_seeds=list(range(8)),
)
lab_manager_plan = preview_lab_manager_training(lab_manager_config, layout=lab_manager_layout)
lab_manager_plan


In [ ]:
if RUN_REAL_TRAINING:
    scientist_result = train_scientist_grpo(scientist_config, layout=scientist_layout)
    lab_manager_result = train_lab_manager_sft(lab_manager_config, layout=lab_manager_layout)
    print(scientist_result)
    print(lab_manager_result)
else:
    print("Dry-run mode is active. Set RUN_REAL_TRAINING = True after reviewing the plans.")


In [ ]:
cases = build_default_evaluation_cases(seeds=[101, 102])
baseline_records, baseline_summary = evaluate_policy(
    base_url=REPLICALAB_URL,
    policy_fn=build_baseline_scientist_action,
    cases=cases,
)
baseline_summary


## Northflank notes

The same module code can be executed in a Northflank GPU job.

Recommended runtime state:
- mount the persistent volume at `REPLICALAB_PERSIST_ROOT`
- set `HF_TOKEN`
- set `ANTHROPIC_API_KEY`
- clone the repo and run the same training functions through `replicalab-train` or this notebook driver

This notebook stays as the readable, judged artifact while the heavy runs happen on H100.
